# Step 1: see the heartbeat

Plan: find the face -> pick skin patches -> average the color -> plot it.
Cells marked **YOUR TURN** are yours. Everything else already works. Run cells top to bottom (Shift+Enter).

In [ ]:
import sys
sys.path.insert(0, "../src")   # so we can import our own files from src/

import cv2
import numpy as np
import matplotlib.pyplot as plt

from ubfc import iter_frames, subject_paths, load_ground_truth, get_fps
from face import make_landmarker, get_landmarks

video, gt_file = subject_paths(1)
print("fps:", get_fps(video))

## 1. Grab one frame and find the face

`next(iter_frames(video))` = just the first frame. `pts` is a table with 478 rows, one per landmark, columns are (x, y) in pixels.

In [ ]:
frame = next(iter_frames(video))
landmarker = make_landmarker()
pts = get_landmarks(landmarker, frame, 0)

print("frame shape:", frame.shape)     # height, width, 3 colors
print("pts shape:", pts.shape)
print("landmark 10 (forehead top) is at x,y =", pts[10])

## 2. What is a mask?

A mask is a black image the same size as the frame. Where we paint white (255), those pixels are "in the region". Everything else is 0 (ignored).
Here is a tiny working example: a triangle mask around the nose. Just run it and look.

In [ ]:
# 4, 129, 358 are three landmarks around the nose (just for the demo)
triangle = pts[[4, 129, 358]].astype(np.int32)   # fillConvexPoly wants whole-number corners

demo_mask = np.zeros(frame.shape[:2], dtype=np.uint8)   # all black, height x width
cv2.fillConvexPoly(demo_mask, triangle, 255)             # paint the triangle white

plt.imshow(demo_mask, cmap="gray"); plt.title("demo mask"); plt.show()
print("pixels inside the mask:", (demo_mask > 0).sum())

## YOUR TURN A: pick the skin regions

Open `outputs/landmarks_frame0.png`. Choose landmark numbers that go around the edge of each patch, in order (like connecting dots).
Rules: skin only. No eyes, eyebrows, mouth, hair.
Keep each patch simple (4-6 points).

Replace the `[]` with your numbers, e.g. `[10, 20, 30, 40]`.

In [ ]:
FOREHEAD    = []   # <- your numbers
LEFT_CHEEK  = []   # <- your numbers (subject's left = right side of the picture)
RIGHT_CHEEK = []   # <- your numbers

Check your picks: this draws your patches on the face. If a patch covers an eye or hair, go back and change the numbers.

In [ ]:
check = frame.copy()
for region in [FOREHEAD, LEFT_CHEEK, RIGHT_CHEEK]:
    if region:
        cv2.polylines(check, [pts[region].astype(np.int32)], True, (0, 255, 0), 1)

x0, y0 = pts.min(axis=0).astype(int) - 10
x1, y1 = pts.max(axis=0).astype(int) + 10
plt.figure(figsize=(6, 6))
plt.imshow(cv2.cvtColor(check[y0:y1, x0:x1], cv2.COLOR_BGR2RGB))   # matplotlib wants RGB, opencv gives BGR
plt.show()

## YOUR TURN B: make a mask from your points

Copy the demo above, but for any list of landmark numbers. Fill the `...` bits.
Hints:
- the corners are `pts[indices].astype(np.int32)`
- start from `np.zeros(shape, dtype=np.uint8)`
- paint with `cv2.fillConvexPoly(mask, corners, 255)`

In [ ]:
def region_mask(shape, pts, indices):
    corners = ...
    mask = ...
    ...
    return mask

## YOUR TURN C: average the color inside a mask

`frame[mask > 0]` picks only the masked pixels, as a table with one row per pixel and 3 columns (B, G, R).
Averaging down the rows = `.mean(axis=0)`.
Careful: opencv order is **B, G, R**. Return them as R, G, B.

In [ ]:
def mean_rgb(frame_bgr, mask):
    pixels = ...
    b, g, r = ...
    return r, g, b

Test both on frame 0. Combine the 3 regions into one mask, then average it.

In [ ]:
full_mask = np.zeros(frame.shape[:2], dtype=np.uint8)
for region in [FOREHEAD, LEFT_CHEEK, RIGHT_CHEEK]:
    full_mask = np.maximum(full_mask, region_mask(frame.shape[:2], pts, region))

print("skin pixels:", (full_mask > 0).sum())
print("mean R, G, B:", mean_rgb(frame, full_mask))